# Custom NISAR GUNW from two local RSLCs (ISCE3 + PLAnT-ISCE3)

This notebook makes a **NISAR L2 GUNW** (geocoded unwrapped interferogram) from two RSLCs on
your disk. It uses the **standard NISAR GUNW processing parameters**: the same ISCE3 InSAR
workflow and settings the NISAR science data system used for an official GUNW (parameter set
`solid_earth_cpu_20_05`, 20+5 MHz mode).

**What does what**

* **ISCE3** `nisar.workflows.insar` does all of the processing. This is the same code NISAR
  runs in production to make RIFG → RUNW → GUNW. It covers the following steps:
  * geometry coregistration
  * dense offsets and rubbersheeting
  * fine resampling
  * interferogram formation (crossmul)
  * phase unwrapping with bridging
  * split-spectrum ionosphere estimation
  * troposphere (optional)
  * solid earth tides
  * geocoding
* **PLAnT-ISCE3** is used to inspect the input RSLCs. It does not produce InSAR products and
  cannot read GUNWs, so the output is examined with h5py, matplotlib and GDAL.

**Standard parameters** (from `../config/nisar_gunw_solid_earth_cpu_20_05.yaml`)

| Step | Setting |
|---|---|
| Input | frequency A, HH |
| Dense offsets | 64 × 96 windows, ±32 search, 40 × 75 skip |
| Rubbersheet | polynomial fit, degree 2 |
| Fine resampling | on |
| Crossmul | 3 range × 6 azimuth looks |
| Unwrapping | 7 × 16 looks, subswath-mask preprocessing, bridging |
| Ionosphere | `main_diff_ms_band` (main band − 5 MHz side band), coherence/subswath-masked filter |
| Troposphere | RAiDER with ECMWF HRES |
| Geocoding | 80 m unwrapped / 20 m wrapped (bilinear) |

**What changes for a custom pair**

* The official run used a fixed geocoding box for its track/frame. Here the box is the whole
  reference scene, in the projection NISAR uses for that location: UTM zone, or polar
  stereographic beyond ±75°.
* Ionosphere correction needs frequency B (the 5 MHz side band) in both RSLCs. If it's missing,
  that step is switched off and the notebook says so.
* Troposphere correction runs only if you provide ECMWF HRES weather-model files and have
  RAiDER installed. Otherwise it is switched off.

**Runtime**: a full NISAR frame takes hours on CPU. Dense offsets is the slowest step, and
`USE_GPU = True` uses ISCE3's CUDA version if your ISCE3 build has it. If a run stops, re-run
it with `RESTART = False` and it continues from the last completed step.

In [ ]:
import argparse
import copy
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import shapely
from osgeo import gdal, osr
from ruamel.yaml import YAML

import isce3
import nisar.workflows
import plant_isce3

gdal.UseExceptions()
print("ISCE3      ", isce3.__version__)
_v = tuple(int(x) for x in re.findall(r"\d+", isce3.__version__)[:2])
if _v < (0, 25):
    raise RuntimeError(f"ISCE3 {isce3.__version__} is too old: the standard NISAR GUNW parameters "
                       "(polynomial rubbersheet, unwrap bridging, subswath masks) need ISCE3 >= 0.25. "
                       "Update with: conda/pixi install 'isce3>=0.25.22'")
print("PLAnT-ISCE3", getattr(plant_isce3, "__version__", "?"))
print("GDAL       ", gdal.__version__)

# A Jupyter kernel started without activating its conda environment can miss PROJ's database.
if "PROJ_DATA" not in os.environ and (Path(sys.prefix) / "share" / "proj").is_dir():
    os.environ["PROJ_DATA"] = str(Path(sys.prefix) / "share" / "proj")

## 1. Inputs

The reference should be the earlier acquisition. Both RSLCs must be from the same track, pass
direction and frame, as for any NISAR GUNW.

Ancillary files:

* **DEM**: leave `DEM_FILE = None` to download the **NISAR DEM v1.2**, the same DEM the
  official products use. This needs an Earthdata Login (https://urs.earthdata.nasa.gov) and
  covers the union of both footprints plus 5 km. You can also point `DEM_FILE` at any GDAL DEM
  with heights above the WGS84 ellipsoid.
* **Water mask, TEC file**: optional. Official runs use them, but the standard parameters
  don't need them to produce a GUNW.
* **Weather-model files** (ECMWF HRES, one per date): needed only for the troposphere layers.

In [ ]:
REF_RSLC = Path("/path/to/NISAR_L1_PR_RSLC_..._reference.h5")    # earlier date
SEC_RSLC = Path("/path/to/NISAR_L1_PR_RSLC_..._secondary.h5")    # later date

WORK_DIR = Path("gunw_work").resolve()                  # runconfig, DEM, scratch and output go here
PARAMETER_FILE = Path("../config/nisar_gunw_solid_earth_cpu_20_05.yaml").resolve()
POLARIZATION = "HH"

DEM_FILE = None                    # None -> download NISAR DEM v1.2 for the scene
WATER_MASK_FILE = None             # optional
TEC_FILE = None                    # optional
REF_WEATHER_MODEL_FILE = None      # optional, enables troposphere (with SEC_ and RAiDER)
SEC_WEATHER_MODEL_FILE = None

OUTPUT_EPSG = None                 # None -> NISAR convention (UTM zone / polar stereographic)
GEOCODE_BOUNDS = None              # None -> whole scene; or (x_min, y_min, x_max, y_max) in OUTPUT_EPSG

USE_GPU = False
RESTART = True                     # False -> resume a previous run from its last completed step

# Changes to the standard parameters, as dotted paths under runconfig.groups, e.g.
# {"processing.phase_unwrap.range_looks": 5}. Leave empty for the standard GUNW.
OVERRIDES = {}

WORK_DIR.mkdir(parents=True, exist_ok=True)
for p in (REF_RSLC, SEC_RSLC, PARAMETER_FILE):
    assert p.is_file(), f"not found: {p}"

## 2. Inspect the RSLCs with PLAnT-ISCE3

`plant_isce3_info` summarises each product: acquisition times, frequencies and polarizations,
bandwidths, spacing, footprint, and the closest NISAR projection.

In [ ]:
def plant_isce3_info(path):
    exe = shutil.which("plant_isce3_info") or str(Path(sys.executable).with_name("plant_isce3_info"))
    res = subprocess.run([exe, str(path)], capture_output=True, text=True)
    return re.sub(r"\x1b\[[0-9;]*m", "", res.stdout + res.stderr)   # strip colour codes

for path in (REF_RSLC, SEC_RSLC):
    print(plant_isce3_info(path))

In [ ]:
def rslc_summary(path):
    """The fields of an RSLC that matter for pairing and for the runconfig."""
    with h5py.File(path, "r") as f:
        band = "LSAR" if "LSAR" in f["science"] else "SSAR"
        root = f[f"science/{band}"]
        ident = root["identification"]
        swaths = (root["RSLC"] if "RSLC" in root else root["SLC"])["swaths"]

        def get(key):
            if key not in ident:
                return None
            v = ident[key][()]
            if isinstance(v, bytes):
                return v.decode()
            if isinstance(v, np.ndarray):
                v = v.ravel()
                return v[0].decode() if v.size == 1 and isinstance(v[0], bytes) else v.tolist()
            return v.item() if hasattr(v, "item") else v

        freqs = {}
        for name in swaths:
            if name.startswith("frequency"):
                g = swaths[name]
                pols = [p.decode() for p in g["listOfPolarizations"][()]]
                freqs[name[-1]] = dict(
                    pols=pols,
                    bandwidth_mhz=round(float(g["processedRangeBandwidth"][()]) / 1e6, 2),
                    center_frequency=float(g["processedCenterFrequency"][()]),
                    shape=g[pols[0]].shape if pols and pols[0] in g else None)
        start = get("zeroDopplerStartTime")
        return dict(
            file=Path(path).name, band=band,
            track=get("trackNumber"), frame=get("frameNumber"),
            pass_direction=str(get("orbitPassDirection")).lower(),
            look=str(get("lookDirection")).lower(),
            start=start, start_dt=datetime.fromisoformat(start[:26]),
            freqs=freqs, footprint=shapely.from_wkt(get("boundingPolygon")))

ref, sec = rslc_summary(REF_RSLC), rslc_summary(SEC_RSLC)

problems, notes = [], []
for key in ("band", "track", "pass_direction", "look"):
    if ref[key] != sec[key]:
        problems.append(f"{key} differs: {ref[key]} vs {sec[key]}")
if ref["frame"] != sec["frame"]:
    notes.append(f"frames differ ({ref['frame']} vs {sec['frame']}); only the overlap is useful")
for s in (ref, sec):
    if POLARIZATION not in s["freqs"].get("A", {}).get("pols", []):
        problems.append(f"{s['file']} has no frequency A {POLARIZATION}")
bw = [s["freqs"]["A"]["bandwidth_mhz"] for s in (ref, sec)]
if bw[0] != bw[1]:
    notes.append(f"frequency A bandwidths differ ({bw[0]} vs {bw[1]} MHz)")
if bw[0] != 20:
    notes.append(f"the standard parameter set is for 20 MHz frequency A; this pair is {bw[0]} MHz")
if sec["start_dt"] <= ref["start_dt"]:
    notes.append("the secondary is not later than the reference; usually you want ref = earlier date")
overlap = ref["footprint"].intersection(sec["footprint"]).area / ref["footprint"].area
if overlap == 0:
    problems.append("the footprints do not overlap")

HAS_SIDE_BAND = all(POLARIZATION in s["freqs"].get("B", {}).get("pols", []) for s in (ref, sec))
HAS_TROPO_INPUTS = bool(REF_WEATHER_MODEL_FILE and SEC_WEATHER_MODEL_FILE)

for label, s in (("reference", ref), ("secondary", sec)):
    fr = ", ".join(f"{k}: {v['pols']} {v['bandwidth_mhz']} MHz" for k, v in s["freqs"].items())
    print(f"{label:9s} {s['file']}\n          track {s['track']} frame {s['frame']} "
          f"{s['pass_direction']} {s['look']}-looking, start {s['start']}\n          {fr}")
print(f"\ntemporal baseline: {(sec['start_dt'] - ref['start_dt']).total_seconds() / 86400:.1f} days, "
      f"footprint overlap: {100 * overlap:.0f} %")
print(f"frequency B {POLARIZATION} in both (needed for ionosphere): {HAS_SIDE_BAND}")
for n in notes:
    print("NOTE:", n)
if problems:
    raise ValueError("These RSLCs can't be paired:\n  " + "\n  ".join(problems))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for s, c in ((ref, "tab:blue"), (sec, "tab:orange")):
    ax.fill(*s["footprint"].exterior.xy, alpha=0.3, color=c, label=s["start"][:10])
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.set_aspect("equal", adjustable="datalim")
ax.legend(); ax.set_title("RSLC footprints");

## 3. DEM

The DEM is downloaded with [sardem](https://github.com/scottstanie/sardem) for the union of
both footprints plus 5 km, as a GeoTIFF with heights above the WGS84 ellipsoid. Choose the
source with `DEM_SOURCE`:

* `"NISAR"`: the NISAR DEM v1.2, the DEM official GUNWs use. It needs an Earthdata Login in
  `~/.netrc`; if there is none, `earthaccess` asks once and saves it.
* `"COP"`: Copernicus GLO-30 from AWS open data, no login needed. sardem converts it to
  ellipsoid heights. It is the same underlying DEM as the NISAR one, but a different release
  and without NISAR's ocean gap filling.

In [ ]:
DEM_SOURCE = "NISAR"     # or "COP" (no login)

DEM_DESCRIPTIONS = {
    "NISAR": ("NISAR DEM v1.2: Copernicus DEM GLO-30 (2023_1) referenced to the WGS84 "
              "ellipsoid, https://doi.org/10.5067/NIDEM-1"),
    "COP": ("Copernicus DEM GLO-30 (AWS open data) converted from EGM2008 to WGS84 "
            "ellipsoid heights with sardem"),
}


def ensure_earthdata_netrc():
    netrc = Path.home() / ".netrc"
    if netrc.exists() and "urs.earthdata.nasa.gov" in netrc.read_text():
        return
    import earthaccess   # conda install -c conda-forge earthaccess
    earthaccess.login(strategy="interactive", persist=True)


def dem_bounds(footprints, margin_km=5.0):
    """(west, south, east, north) of the union of the footprints plus a margin."""
    min_x, min_y, max_x, max_y = shapely.union_all(footprints).bounds
    if max_x - min_x > 180:
        raise NotImplementedError("scene crosses the antimeridian; provide DEM_FILE yourself")
    lat = max(abs(min_y), abs(max_y))
    d_lat = margin_km / 111.0
    d_lon = margin_km / (111.0 * max(np.cos(np.deg2rad(lat)), 0.05))
    return (min_x - d_lon, max(min_y - d_lat, -90), max_x + d_lon, min(max_y + d_lat, 90))


def stage_dem(footprints, out_path, source=DEM_SOURCE, margin_km=5.0):
    import sardem.dem   # pip install sardem
    if source == "NISAR":
        ensure_earthdata_netrc()
    tmp = Path(str(out_path) + ".part.tif")     # only rename once complete
    sardem.dem.main(output_name=str(tmp), bbox=dem_bounds(footprints, margin_km),
                    data_source=source, output_format="GTiff", output_type="float32")
    tmp.replace(out_path)
    return Path(out_path)


if DEM_FILE is None:
    dem_path = WORK_DIR / f"dem_{DEM_SOURCE.lower()}.tif"
    if not dem_path.exists():
        t0 = time.time()
        stage_dem([ref["footprint"], sec["footprint"]], dem_path)
        print(f"downloaded DEM in {time.time() - t0:.0f} s")
    DEM_PATH, DEM_DESCRIPTION = dem_path, DEM_DESCRIPTIONS[DEM_SOURCE]
else:
    DEM_PATH, DEM_DESCRIPTION = Path(DEM_FILE).resolve(), f"user-provided DEM {Path(DEM_FILE).name}"

info = gdal.Info(str(DEM_PATH), format="json")
print(DEM_PATH, info["size"], "EPSG", osr.SpatialReference(info["coordinateSystem"]["wkt"]).GetAuthorityCode(None))

## 4. Build the runconfig

The standard parameters are loaded and the pair-specific fields are filled in: inputs, DEM,
output paths, projection, and which optional steps can run. The cell then lists every
processing setting that differs from the standard set, so you can see exactly how this
custom GUNW departs from an official one.

In [ ]:
def point_to_nisar_epsg(lon, lat):
    """Projection NISAR uses for geocoded products at (lon, lat): ISCE3's own rule."""
    from nisar.workflows.stage_dem import point2epsg
    return point2epsg(lon, lat)


def set_dotted(tree, dotted, value):
    keys = dotted.split(".")
    for k in keys[:-1]:
        if tree.get(k) is None:
            tree[k] = {}
        tree = tree[k]
    tree[keys[-1]] = value


def flatten(tree, prefix=""):
    out = {}
    for k, v in (tree or {}).items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            out.update(flatten(v, key + "."))
        else:
            out[key] = v
    return out


yaml = YAML()
yaml.default_flow_style = False
standard = yaml.load(PARAMETER_FILE)
rc = copy.deepcopy(standard)
groups = rc["runconfig"]["groups"]
proc = groups["processing"]

out_dir, scratch_dir, qa_dir = WORK_DIR / "output", WORK_DIR / "scratch", WORK_DIR / "qa"
gunw_name = (f"GUNW_{ref['band']}_T{int(ref['track']):03d}_F{int(ref['frame']):03d}_"
             f"{ref['start_dt']:%Y%m%dT%H%M%S}_{sec['start_dt']:%Y%m%dT%H%M%S}_{POLARIZATION}.h5")
GUNW_PATH = out_dir / gunw_name
LOG_PATH = scratch_dir / "insar.log"
for d in (out_dir, scratch_dir, qa_dir):
    d.mkdir(parents=True, exist_ok=True)

groups["input_file_group"] = {
    "reference_rslc_file": str(REF_RSLC.resolve()),
    "secondary_rslc_file": str(SEC_RSLC.resolve()),
    "qa_gunw_input_file": str(GUNW_PATH)}
groups["dynamic_ancillary_file_group"] = {
    "dem_file": str(DEM_PATH),
    "dem_file_description": DEM_DESCRIPTION,
    "water_mask_file": str(Path(WATER_MASK_FILE).resolve()) if WATER_MASK_FILE else None,
    "orbit_files": {"reference_orbit_file": None, "secondary_orbit_file": None},
    "troposphere_weather_model_files": {
        "reference_troposphere_file": str(Path(REF_WEATHER_MODEL_FILE).resolve()) if REF_WEATHER_MODEL_FILE else None,
        "secondary_troposphere_file": str(Path(SEC_WEATHER_MODEL_FILE).resolve()) if SEC_WEATHER_MODEL_FILE else None},
    "tec_file": str(Path(TEC_FILE).resolve()) if TEC_FILE else None}
groups["product_path_group"] = {
    "product_path": str(out_dir), "scratch_path": str(scratch_dir),
    "sas_output_file": str(GUNW_PATH), "product_counter": 1, "qa_output_dir": str(qa_dir)}
groups["primary_executable"]["processing_type"] = "OD"       # on-demand (custom) processing
groups["logging"] = {"path": str(LOG_PATH), "write_mode": "w" if RESTART else "a"}

# Polarization and projection.
proc["input_subset"]["list_of_frequencies"] = {"A": [POLARIZATION]}
center = shapely.union_all([ref["footprint"], sec["footprint"]]).centroid
epsg = int(OUTPUT_EPSG or point_to_nisar_epsg(center.x, center.y))
geo = proc["geocode"]
geo["output_epsg"] = epsg
if GEOCODE_BOUNDS:
    x0, y0, x1, y1 = GEOCODE_BOUNDS
    geo["top_left"], geo["bottom_right"] = {"x_abs": x0, "y_abs": y1}, {"x_abs": x1, "y_abs": y0}
else:
    geo["top_left"], geo["bottom_right"] = {"x_abs": None, "y_abs": None}, {"x_abs": None, "y_abs": None}
proc["radar_grid_cubes"] = {
    "output_epsg": epsg,
    "output_posting": {"x_posting": None, "y_posting": None},
    "top_left": {"x_abs": None, "y_abs": None},
    "bottom_right": {"x_abs": None, "y_abs": None}}

# Optional steps that depend on the data / ancillary files available.
skipped = []
if not HAS_SIDE_BAND:
    proc["ionosphere_phase_correction"]["enabled"] = False
    skipped.append(f"ionosphere: frequency B {POLARIZATION} is not in both RSLCs")
try:
    import RAiDER  # noqa: F401
    has_raider = True
except ImportError:
    has_raider = False
if not (HAS_TROPO_INPUTS and has_raider):
    proc["troposphere_delay"]["enabled"] = False
    skipped.append("troposphere: " + ("no weather-model files given" if not HAS_TROPO_INPUTS
                                      else "RAiDER is not installed"))
groups["worker"]["gpu_enabled"] = bool(USE_GPU)

for dotted, value in OVERRIDES.items():
    set_dotted(groups, dotted, value)

RUNCONFIG = WORK_DIR / "runconfig_gunw.yaml"
with open(RUNCONFIG, "w") as fh:
    yaml.dump(rc, fh)

print(f"runconfig : {RUNCONFIG}\noutput    : {GUNW_PATH}\nEPSG      : {epsg}")
for s in skipped:
    print("SKIPPED   :", s)
std, new = flatten(standard["runconfig"]["groups"]["processing"]), flatten(proc)
# (empty values mean "ISCE3 default", so they are not differences)
diff = {k: (std.get(k, "<ISCE3 default>"), new[k]) for k in new
        if new[k] is not None and std.get(k, object()) != new[k]}
diff.update({k: (std[k], "<removed>") for k in std if k not in new})
print("\nProcessing settings that differ from the standard parameter set:")
for k, (a, b) in sorted(diff.items()):
    print(f"  {k:70s} {a!s:>16s} -> {b}")

In [ ]:
# Check the runconfig against ISCE3's schema and its own consistency checks before
# starting a long run. ISCE3 0.25 can hide schema errors behind a crash in its own error
# reporting, so validate with yamale first to get readable messages.
import yamale

schema = yamale.make_schema(Path(nisar.workflows.__file__).parent / "schemas" / "insar.yaml", parser="ruamel")
try:
    yamale.validate(schema, yamale.make_data(str(RUNCONFIG), parser="ruamel"))
except yamale.YamaleError as err:
    raise ValueError("runconfig does not match the ISCE3 schema:\n" +
                     "\n".join(e for r in err.results for e in r.errors)) from None

from nisar.workflows.insar_runconfig import InsarRunConfig
InsarRunConfig(argparse.Namespace(run_config_path=str(RUNCONFIG), log_file=False))
print("runconfig is valid for ISCE3", isce3.__version__)

## 5. Run the ISCE3 InSAR workflow

This runs the same command as NISAR production, `python -m nisar.workflows.insar
runconfig.yaml`, as a separate process. Each step is printed as it finishes. The full log is
`scratch/insar.log`, and the program's own output goes to `insar_stdout.log` in `WORK_DIR`.
Intermediate files are removed as steps complete (`intermediate_files_removal_enabled`, as in
production).

In [ ]:
def run_insar(runconfig, log_path, restart=True, poll_s=5):
    cmd = [sys.executable, "-m", "nisar.workflows.insar", str(runconfig)] + (["--restart"] if restart else [])
    stdout_path = WORK_DIR / "insar_stdout.log"
    seen = len(log_path.read_text(errors="replace").splitlines()) if (log_path.exists() and not restart) else 0
    t0 = time.time()
    with open(stdout_path, "w") as out:
        p = subprocess.Popen(cmd, stdout=out, stderr=subprocess.STDOUT, cwd=WORK_DIR, env=os.environ.copy())
        while True:
            done = p.poll() is not None
            if log_path.exists():
                lines = log_path.read_text(errors="replace").splitlines()
                for line in lines[seen:]:
                    if re.search(r"[Ss]uccessfully ran", line):
                        print(f"[{(time.time() - t0) / 60:6.1f} min] {line.strip(' -')}")
                seen = len(lines)
            if done:
                break
            time.sleep(poll_s)
    if p.returncode != 0:
        tail = stdout_path.read_text(errors="replace").splitlines()[-40:]
        raise RuntimeError(f"ISCE3 InSAR workflow failed (exit {p.returncode}); "
                           f"last lines of {stdout_path}:\n" + "\n".join(tail))
    print(f"done in {(time.time() - t0) / 60:.1f} min -> {GUNW_PATH}")


run_insar(RUNCONFIG, LOG_PATH, restart=RESTART)

## 6. Look at the GUNW

The GUNW holds, on the geocoded grid:

* `wrappedInterferogram` at 20 m: complex interferogram and coherence
* `unwrappedInterferogram` at 80 m: unwrapped phase, coherence, connected components,
  ionosphere phase screen
* `pixelOffsets` at 80 m

It also has the radar-grid cubes and full processing metadata. The runconfig used to make it
is stored inside the product.

In [ ]:
GRIDS = f"science/{ref['band']}/GUNW/grids/frequencyA"

with h5py.File(GUNW_PATH, "r") as f:
    ident = f[f"science/{ref['band']}/identification"]
    for k in ("productType", "referenceZeroDopplerStartTime", "secondaryZeroDopplerStartTime",
              "trackNumber", "frameNumber", "orbitPassDirection", "processingType", "isGeocoded"):
        if k in ident:
            v = ident[k][()]
            print(f"{k:32s} {v.decode() if isinstance(v, bytes) else v}")
    print("\nlayers:")
    def show(name, obj):
        if isinstance(obj, h5py.Dataset) and obj.ndim == 2:
            print(f"  {name:70s} {str(obj.shape):>14s} {obj.dtype}")
    f[GRIDS].visititems(show)

In [ ]:
def read_layer(path):
    """Data (fill values -> NaN) and the matplotlib extent of a GUNW grid layer."""
    with h5py.File(GUNW_PATH, "r") as f:
        ds = f[f"{GRIDS}/{path}"]
        data = ds[()]
        fill = ds.attrs.get("_FillValue")
        grp = ds.parent.parent
        x, y = grp["xCoordinates"][()], grp["yCoordinates"][()]
        dx, dy = float(grp["xCoordinateSpacing"][()]), float(grp["yCoordinateSpacing"][()])
    if data.dtype.kind in "fc":
        if fill is not None and np.isfinite(fill):
            data = np.where(data == fill, np.nan, data)
    extent = (x[0] - dx / 2, x[-1] + dx / 2, y[-1] + dy / 2, y[0] - dy / 2)
    return data, extent


pol = POLARIZATION
wrapped, ext_w = read_layer(f"wrappedInterferogram/{pol}/wrappedInterferogram")
coh, ext_u = read_layer(f"unwrappedInterferogram/{pol}/coherenceMagnitude")
unw, _ = read_layer(f"unwrappedInterferogram/{pol}/unwrappedPhase")
ccomp, _ = read_layer(f"unwrappedInterferogram/{pol}/connectedComponents")
labelled = (ccomp > 0) & (ccomp != 65535)          # 0 = not in a component, 65535 = fill
if labelled.any():
    unw_masked, unw_title = np.where(labelled, unw, np.nan), "unwrapped phase (rad), connected comps"
else:
    print("note: no pixel is in a connected component; showing the unmasked unwrapped phase")
    unw_masked, unw_title = unw, "unwrapped phase (rad), unmasked"

panels = [
    (np.angle(wrapped), ext_w, "twilight", "wrapped phase (rad), 20 m", (-np.pi, np.pi)),
    (coh, ext_u, "gray", "coherence", (0, 1)),
    (unw_masked - np.nanmedian(unw_masked), ext_u, "RdBu_r", unw_title + ", median removed", None),
    (np.where(ccomp == 65535, np.nan, ccomp), ext_u, "tab20", "connected components (0 = none)", (0, max(1, int(ccomp[ccomp != 65535].max(initial=0))))),
]
iono_path = f"unwrappedInterferogram/{pol}/ionospherePhaseScreen"
with h5py.File(GUNW_PATH, "r") as f:
    has_iono = iono_path in f[GRIDS]
if has_iono:
    iono, _ = read_layer(iono_path)
    if np.isfinite(iono).any():
        panels.append((iono - np.nanmedian(iono), ext_u, "RdBu_r", "ionosphere phase screen (rad)", None))

fig, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 5), constrained_layout=True)
for ax, (img, extent, cmap, title, clim) in zip(np.atleast_1d(axes), panels):
    if clim is None:
        clim = np.nanpercentile(img, [2, 98]) if np.isfinite(img).any() else (0, 1)
    im = ax.imshow(img, extent=extent, cmap=cmap, vmin=clim[0], vmax=clim[1], interpolation="nearest")
    ax.set_title(title, fontsize=10); ax.set_xlabel(f"x (EPSG:{epsg})"); ax.ticklabel_format(useOffset=False, style="plain")
    fig.colorbar(im, ax=ax, shrink=0.7)
np.atleast_1d(axes)[0].set_ylabel(f"y (EPSG:{epsg})");

In [ ]:
# Line-of-sight displacement from the unwrapped phase. NISAR convention: positive phase =
# range increase (motion away from the satellite). Uses the processed centre frequency of A.
wavelength = isce3.core.speed_of_light / ref["freqs"]["A"]["center_frequency"]
los_cm = -unw_masked * wavelength / (4 * np.pi) * 100
los_cm -= np.nanmedian(los_cm)
assert np.isfinite(los_cm).any(), "no valid unwrapped pixels"
print(f"wavelength {wavelength * 100:.2f} cm; LOS displacement 2-98 %: "
      f"{np.nanpercentile(los_cm, 2):.2f} .. {np.nanpercentile(los_cm, 98):.2f} cm (relative)")

## 7. Export layers to GeoTIFF

The GUNW is a single HDF5 file. These GeoTIFFs make the main layers easy to open in GIS tools.

In [ ]:
def gunw_layer_to_geotiff(path, out_tif):
    data, (x0, x1, y0, y1) = read_layer(path)
    if np.iscomplexobj(data):
        data = np.angle(data).astype(np.float32)       # wrapped phase
    ny, nx = data.shape
    dtype = gdal.GDT_Float32 if data.dtype.kind == "f" else gdal.GDT_UInt16
    ds = gdal.GetDriverByName("GTiff").Create(str(out_tif), nx, ny, 1, dtype,
                                              options=["COMPRESS=DEFLATE", "TILED=YES"])
    ds.SetGeoTransform((x0, (x1 - x0) / nx, 0, y1, 0, (y0 - y1) / ny))
    srs = osr.SpatialReference(); srs.ImportFromEPSG(epsg)
    ds.SetProjection(srs.ExportToWkt())
    band = ds.GetRasterBand(1)
    band.WriteArray(data)
    if data.dtype.kind == "f":
        band.SetNoDataValue(float("nan"))
    ds.FlushCache()
    return Path(out_tif)


export_dir = WORK_DIR / "geotiff"
export_dir.mkdir(exist_ok=True)
stem = GUNW_PATH.stem
layers = {"unwrappedPhase": f"unwrappedInterferogram/{pol}/unwrappedPhase",
          "coherence": f"unwrappedInterferogram/{pol}/coherenceMagnitude",
          "connectedComponents": f"unwrappedInterferogram/{pol}/connectedComponents",
          "wrappedPhase": f"wrappedInterferogram/{pol}/wrappedInterferogram"}
if has_iono:
    layers["ionospherePhaseScreen"] = iono_path
for name, path in layers.items():
    print(gunw_layer_to_geotiff(path, export_dir / f"{stem}_{name}.tif"))

## Notes

* **Is it standard?** The settings listed in step 4 as differing from the standard set are the
  only processing differences from an official GUNW made with the same ISCE3 version. The
  runconfig is stored in the product at
  `science/LSAR/GUNW/metadata/processingInformation/parameters/runConfigurationContents`.
* **Ancillary data:** official GUNWs also use the NISAR water mask, a TEC file and ECMWF HRES
  weather models. Add them under section 1 if you have them.
* **QA:** the `qa` section of the parameter file holds NISAR's QA settings. To produce the
  official QA browse images and reports, run the separate NISAR QA package
  (`nisarqa`) on `GUNW_PATH`.
* **Restarting:** a run that stopped part-way can continue with `RESTART = False`.
  `OVERRIDES` changes individual parameters without editing the parameter file.